# Domain 2: Tool Design & MCP Integration
## Claude Certified Architect – Foundations Certification
**Weight: 18% of scored content**

This notebook covers designing effective tool interfaces, MCP (Model Context Protocol) integration,
error handling, tool distribution, and built-in tool usage.

---

## Task 2.1: Design Effective Tool Interfaces with Clear Descriptions and Boundaries

### Why Tool Descriptions Matter

Tool descriptions are the **primary mechanism** LLMs use for tool selection.
Minimal descriptions → unreliable selection among similar tools.

### Common Problems

| Problem | Example |
|---|---|
| **Ambiguous descriptions** | `analyze_content` vs `analyze_document` with near-identical descriptions |
| **Missing boundaries** | No explanation of when to use Tool A vs Tool B |
| **System prompt conflicts** | Keyword-sensitive instructions override well-written tool descriptions |

### Best Practices

1. Include: input formats, example queries, edge cases, and boundary explanations
2. Clearly differentiate each tool's purpose vs similar alternatives
3. Rename and update descriptions to eliminate functional overlap
4. Split generic tools into purpose-specific tools

In [ ]:
# === TOOL DESCRIPTION DESIGN ===

# ❌ BAD: Minimal descriptions that cause confusion
bad_tools = [
    {
        "name": "get_customer",
        "description": "Retrieves customer information",  # Too vague!
        "input_schema": {
            "type": "object",
            "properties": {"id": {"type": "string"}},
            "required": ["id"]
        }
    },
    {
        "name": "lookup_order",
        "description": "Retrieves order details",  # Too vague!
        "input_schema": {
            "type": "object",
            "properties": {"id": {"type": "string"}},
            "required": ["id"]
        }
    }
]

# ✅ GOOD: Detailed descriptions with boundaries and examples
good_tools = [
    {
        "name": "get_customer",
        "description": (
            "Look up a customer's account by their customer ID, email address, or phone number. "
            "Returns: customer name, account status, contact info, and verification status. "
            "Use this FIRST to verify the customer's identity before any order operations. "
            "Input: customer_id (e.g., 'CUST-12345'), email, or phone. "
            "Do NOT use for order lookups — use lookup_order instead."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Customer ID (format: CUST-XXXXX)"},
                "email": {"type": "string", "description": "Customer email address"},
                "phone": {"type": "string", "description": "Customer phone number"}
            }
        }
    },
    {
        "name": "lookup_order",
        "description": (
            "Retrieve details for a specific order by order ID or tracking number. "
            "Returns: order items, amounts, shipping status, and return eligibility. "
            "Requires a verified customer ID from get_customer. "
            "Input: order_id (e.g., 'ORD-98765') or tracking_number. "
            "Use get_customer first if you only have a customer name or email."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID (format: ORD-XXXXX)"},
                "tracking_number": {"type": "string", "description": "Shipping tracking number"}
            }
        }
    }
]

print("=== BAD Tool Descriptions ===")
for t in bad_tools:
    print(f"  {t['name']}: {t['description']}")

print("\n=== GOOD Tool Descriptions ===")
for t in good_tools:
    print(f"  {t['name']}: {t['description'][:80]}...")

### Splitting Generic Tools

**Anti-pattern:** One generic `analyze_document` tool that does everything

**Better:** Split into purpose-specific tools:
- `extract_data_points` — Extract structured data from documents
- `summarize_content` — Generate summaries of document content
- `verify_claim_against_source` — Check if a specific claim is supported by a source

Each tool has a clear, non-overlapping purpose with defined input/output contracts.

---

## Task 2.2: Implement Structured Error Responses for MCP Tools

### The MCP `isError` Flag

MCP tools communicate failures using the `isError` flag. But **not all errors are equal**.

### Error Categories

| Category | Example | Retryable? | Agent Action |
|---|---|---|---|
| **Transient** | Timeout, service unavailable | ✅ Yes | Retry with backoff |
| **Validation** | Invalid input format | ❌ No | Fix input and retry |
| **Business** | Policy violation | ❌ No | Explain to user |
| **Permission** | Unauthorized access | ❌ No | Escalate |

### Anti-Pattern: Uniform Error Responses

❌ Generic `"Operation failed"` — prevents the agent from making appropriate recovery decisions

### Critical Distinction

- **Access failure** (timeout) ≠ **Valid empty result** (query succeeded, no matches)
- These require completely different agent responses!

In [ ]:
# === STRUCTURED MCP ERROR RESPONSES ===

import json
from enum import Enum

class ErrorCategory(Enum):
    TRANSIENT = "transient"      # Timeouts, temporary unavailability
    VALIDATION = "validation"    # Bad input
    BUSINESS = "business"        # Policy violations
    PERMISSION = "permission"    # Unauthorized

def create_tool_error(category: ErrorCategory, message: str, 
                      is_retryable: bool, details: dict = None):
    """
    Structured error response for MCP tools.
    
    Exam-critical: Include errorCategory, isRetryable, and
    human-readable description so the agent can make appropriate
    recovery decisions.
    """
    return {
        "isError": True,
        "errorCategory": category.value,
        "isRetryable": is_retryable,
        "message": message,
        "details": details or {}
    }

def create_tool_success(data, empty_result=False):
    """Successful response — distinguish success from empty result."""
    return {
        "isError": False,
        "data": data,
        "isEmpty": empty_result  # Valid query, no matches found
    }

# Examples of different error types
examples = [
    ("Timeout", create_tool_error(
        ErrorCategory.TRANSIENT,
        "Database query timed out after 30s",
        is_retryable=True,
        details={"attempted_query": "SELECT * FROM orders WHERE id='ORD-123'"}
    )),
    ("Invalid Input", create_tool_error(
        ErrorCategory.VALIDATION,
        "Invalid order ID format. Expected 'ORD-XXXXX'",
        is_retryable=False
    )),
    ("Policy Violation", create_tool_error(
        ErrorCategory.BUSINESS,
        "Refund exceeds maximum automated limit of $500. Customer-friendly explanation: "
        "This refund requires manager approval due to the amount.",
        is_retryable=False,
        details={"requested_amount": 750, "limit": 500}
    )),
    ("Empty Result (Success!)", create_tool_success(
        data=[],
        empty_result=True  # Not an error — query worked, no matches
    ))
]

for name, response in examples:
    print(f"\n{name}:")
    print(f"  isError: {response.get('isError')}")
    if response.get('isError'):
        print(f"  category: {response['errorCategory']}")
        print(f"  retryable: {response['isRetryable']}")
        print(f"  message: {response['message'][:60]}...")
    else:
        print(f"  isEmpty: {response.get('isEmpty')} (valid query, just no data)")

### Error Propagation in Multi-Agent Systems

- Subagents should implement **local error recovery** for transient failures
- Only propagate errors that cannot be resolved locally
- Include partial results and what was attempted

---

## Task 2.3: Distribute Tools Appropriately Across Agents and Configure Tool Choice

### The Tool Overload Problem

> Giving an agent access to 18 tools instead of 4-5 **degrades tool selection reliability**
> by increasing decision complexity.

### Principles

1. **Scope tools to roles** — Each subagent gets only tools relevant to its role
2. **Prevent cross-specialization misuse** — A synthesis agent shouldn't have web search tools
3. **Scoped cross-role tools** — For high-frequency needs (e.g., `verify_fact` for synthesis agent)

### `tool_choice` Configuration

| Setting | Behavior |
|---|---|
| `"auto"` | Model decides whether/which tool to call (may return text instead) |
| `"any"` | Model MUST call a tool (guarantees structured output) |
| `{"type": "tool", "name": "..."}` | Forces a specific tool to be called first |

In [ ]:
# === TOOL DISTRIBUTION AND TOOL_CHOICE ===

# Agent role definitions with scoped tools
agent_tool_configs = {
    "search_agent": {
        "description": "Searches the web for information",
        "allowed_tools": ["web_search", "fetch_url"],  # Only search tools
        "tool_choice": "auto"  # Model decides which search tool
    },
    "analysis_agent": {
        "description": "Analyzes documents and data",
        "allowed_tools": ["extract_data_points", "summarize_content"],
        "tool_choice": "auto"
    },
    "synthesis_agent": {
        "description": "Synthesizes findings into reports",
        "allowed_tools": [
            "synthesize",     # Primary tool
            "verify_fact"     # Scoped cross-role tool for simple lookups
        ],
        "tool_choice": "auto"
    },
    "extraction_agent": {
        "description": "Extracts structured data from documents",
        "allowed_tools": ["extract_metadata", "enrich_data"],
        # Force metadata extraction first, then enrichment in follow-up turns
        "tool_choice": {"type": "tool", "name": "extract_metadata"}
    }
}

# tool_choice examples
tool_choice_examples = {
    '"auto"': 'Model may call a tool OR return text. Use for general conversation.',
    '"any"': 'Model MUST call a tool. Use when you need guaranteed structured output.',
    '{"type": "tool", "name": "extract_metadata"}': (
        'Forces specific tool first. Process subsequent steps in follow-up turns.'
    )
}

print("Agent Tool Distribution:")
for agent, config in agent_tool_configs.items():
    print(f"  {agent}: {len(config['allowed_tools'])} tools - {config['allowed_tools']}")

print("\ntool_choice Options:")
for choice, desc in tool_choice_examples.items():
    print(f"  {choice}")
    print(f"    → {desc}")

---

## Task 2.4: Integrate MCP Servers into Claude Code and Agent Workflows

### MCP Server Scoping

| Scope | Location | Shared? | Use Case |
|---|---|---|---|
| **Project-level** | `.mcp.json` | ✅ Via version control | Shared team tooling |
| **User-level** | `~/.claude.json` | ❌ Personal only | Personal/experimental servers |

### Environment Variable Expansion

```json
// .mcp.json — credentials via env vars (never commit secrets!)
{
  "mcpServers": {
    "github": {
      "command": "mcp-server-github",
      "env": {
        "GITHUB_TOKEN": "${GITHUB_TOKEN}"  // Expanded at runtime
      }
    }
  }
}
```

### MCP Resources vs Tools

| MCP Feature | Purpose | Example |
|---|---|---|
| **Tools** | Perform actions | `create_issue`, `search_code` |
| **Resources** | Expose content catalogs | Issue summaries, documentation hierarchies, DB schemas |

**Resources reduce exploratory tool calls** — agents can browse available data without calling tools.

### Best Practices

- Use **community MCP servers** for standard integrations (Jira, GitHub)
- Reserve **custom servers** for team-specific workflows
- Enhance MCP tool descriptions to prevent agents from preferring built-in tools (like Grep) over more capable MCP tools

In [ ]:
# === MCP SERVER CONFIGURATION ===

import json

# Project-level .mcp.json (shared via version control)
project_mcp_config = {
    "mcpServers": {
        "github": {
            "command": "mcp-server-github",
            "env": {
                "GITHUB_TOKEN": "${GITHUB_TOKEN}"  # Environment variable expansion
            }
        },
        "jira": {
            "command": "mcp-server-jira",
            "env": {
                "JIRA_URL": "${JIRA_URL}",
                "JIRA_TOKEN": "${JIRA_TOKEN}"
            }
        }
    }
}

# User-level ~/.claude.json (personal, not shared)
user_mcp_config = {
    "mcpServers": {
        "experimental-db": {
            "command": "mcp-server-postgres",
            "args": ["--connection-string", "postgresql://localhost/devdb"]
        }
    }
}

print("Project-level .mcp.json (shared with team):")
print(json.dumps(project_mcp_config, indent=2))

print("\nUser-level ~/.claude.json (personal):")
print(json.dumps(user_mcp_config, indent=2))

---

## Task 2.5: Select and Apply Built-in Tools Effectively

### Built-in Tool Selection Guide

| Tool | Purpose | Use When |
|---|---|---|
| **Grep** | Search file contents for patterns | Finding function callers, error messages, import statements |
| **Glob** | Find files by name/extension pattern | Finding files: `**/*.test.tsx`, `src/api/**/*.py` |
| **Read** | Load full file contents | Need to see entire file |
| **Write** | Write entire file | Creating new files or full replacements |
| **Edit** | Targeted modifications | Changing specific sections using unique text matching |
| **Bash** | Run shell commands | Build, test, install, arbitrary commands |

### Key Distinctions

- **Grep** = Search *content* (what's inside files)
- **Glob** = Search *paths* (file names and locations)
- When **Edit** fails (non-unique text matches) → fall back to **Read + Write**

### Incremental Codebase Understanding

```
1. Grep → Find entry points (function names, imports)
2. Read → Follow imports, trace execution flows
3. DON'T read all files upfront!
```

### Tracing Function Usage

```
1. Identify all exported names from a module
2. Grep for each name across the codebase
3. Follow through wrapper modules
```

In [ ]:
# === BUILT-IN TOOL SELECTION DECISION TREE ===

def select_tool(task_description):
    """
    Decision guide for built-in tool selection.
    This mirrors the exam's expected reasoning.
    """
    task = task_description.lower()
    
    if any(kw in task for kw in ["find files", "file pattern", "file name", "extension"]):
        return "Glob", "Find files by name pattern (e.g., **/*.test.tsx)"
    
    elif any(kw in task for kw in ["search content", "find function", "find import", 
                                    "find callers", "grep", "pattern in code"]):
        return "Grep", "Search file contents for patterns"
    
    elif any(kw in task for kw in ["read file", "view file", "see contents"]):
        return "Read", "Load full file contents"
    
    elif any(kw in task for kw in ["create file", "write new"]):
        return "Write", "Create or overwrite a file"
    
    elif any(kw in task for kw in ["change", "modify", "update", "replace"]):
        return "Edit", "Targeted modification using unique text matching"
    
    elif any(kw in task for kw in ["run", "build", "test", "install", "execute"]):
        return "Bash", "Run shell commands"
    
    return "Unknown", "Analyze the task further"

# Test cases
tasks = [
    "Find all test files in the project",
    "Search for all callers of the authenticate() function",
    "Read the main configuration file",
    "Change the database timeout from 30 to 60",
    "Run the test suite",
    "Find files with .tsx extension"
]

for task in tasks:
    tool, reason = select_tool(task)
    print(f"  Task: '{task}'")
    print(f"  → Use: {tool} ({reason})")
    print()

---

## Domain 2 Summary & Exam Preparation

### Key Concepts to Remember

1. **Tool Descriptions** are the #1 mechanism for tool selection — make them detailed
2. **Structured Errors**: Include `errorCategory`, `isRetryable`, and human-readable messages
3. **Access failure ≠ empty result** — these need different handling
4. **Tool count**: 4-5 tools per agent is ideal; 18 degrades reliability
5. **`tool_choice`**: `"auto"` (optional), `"any"` (must call), forced (specific tool)
6. **MCP scoping**: `.mcp.json` (project) vs `~/.claude.json` (user)
7. **MCP Resources** reduce exploratory tool calls
8. **Grep** = content search, **Glob** = file path matching

### Common Exam Traps

> ❌ Consolidating tools to reduce count (may be valid but not always the right "first step")
>
> ❌ Adding a routing layer instead of fixing tool descriptions
>
> ❌ Using uniform error responses ("Operation failed")
>
> ✅ First step is usually: **improve tool descriptions**